# Local neurotransmitter fingerprinting

In this tutorial, you will explore the local neurotransmitter fingerprinting (LNTF) functionalities of Lacuna using the CLI.

**What you'll learn**:

- Fetch the neurotransmitter PET atlas
- Compute per-target neurotransmitter density scores within a lesion
- Filter by neurotransmitter system presets
- Obtain parcel-level NT scores

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-petersen/lacuna/blob/main/docs/tutorials/06-local-neurotransmitter-fingerprinting.ipynb)

## Colab

Note: Colab provides limited computational resources. While these tutorials are designed to operate within those constraints, some Lacuna functionality cannot be fully demonstrated in this environment and requires access to higher-performance computing infrastructure.

Ignore this if you run this notebook locally.

## Setup

In [ ]:
# Install Lacuna from GitHub
!pip install git+https://github.com/m-petersen/lacuna

Get the tutorial data.

In [39]:
# Get tutorial data
!lacuna tutorial /tmp/tutorial_bids --force


Setting up tutorial data at: /tmp/tutorial_bids
✓ Tutorial data copied to: /tmp/tutorial_bids


## Fetch neurotransmitter atlas

Local neurotransmitter fingerprinting requires a neurotransmitter PET atlas: a collection of PET receptor/transporter density maps from normative cohorts. Lacuna downloads representative maps from [NiSpace-data](https://github.com/LeonDLotter/NiSpace-data) — one recommended map per target — at a pinned commit, with SHA-256 verification.

`lacuna fetch ntatlas` downloads, z-scores, and saves a ready-to-use atlas in a single step. The output directory contains the prepared atlas (manifest + per-target NIfTIs) and is consumed directly by `lacuna run lntf`.

In [42]:
!lacuna fetch ntatlas \
    --output-dir /tmp/ntatlas_data --force

Fetching neurotransmitter PET maps...
  Source: https://github.com/LeonDLotter/NiSpace-data
  Commit: a772c410f76dbbd022a3f5239e40176ff6f930c7


✓ Neurotransmitter atlas fetch complete!
  Files: 22
  Duration: 2.3s
  Output: /tmp/ntatlas_data


In [43]:
!ls /tmp/ntatlas_data

manifest.json
maps
target-5HT1a_tracer-way100635_n-35_dx-hc_pub-savli2012_space-MNI152NLin6Asym_desc-proc.nii.gz
target-5HT1b_tracer-p943_n-23_dx-hc_pub-savli2012_space-MNI152NLin6Asym_desc-proc.nii.gz
target-5HT2a_tracer-altanserin_n-19_dx-hc_pub-savli2012_space-MNI152NLin6Asym_desc-proc.nii.gz
target-5HT4_tracer-sb207145_n-59_dx-hc_pub-beliveau2017_space-MNI152NLin6Asym_desc-proc.nii.gz
target-5HT6_tracer-gsk215083_n-30_dx-hc_pub-radhakrishnan2018_space-MNI152NLin6Asym_desc-proc.nii.gz
target-5HTT_tracer-dasb_n-18_dx-hc_pub-savli2012_space-MNI152NLin6Asym_desc-proc.nii.gz
target-A4B2_tracer-flubatine_n-30_dx-hc_pub-hillmer2016_space-MNI152NLin6Asym_desc-proc.nii.gz
target-CB1_tracer-omar_n-77_dx-hc_pub-normandin2015_space-MNI152NLin6Asym_desc-proc.nii.gz
target-D1_tracer-sch23390_n-13_dx-hc_pub-kaller2017_space-MNI152NLin6Asym_desc-proc.nii.gz
target-D23_tracer-flb457_n-55_dx-hc_pub-sandiego2015_space-MNI152NLin6Asym_desc-proc.nii.gz
target-DAT_tracer-fpcit_n-174_dx-hc_pub-dukart2018

## Analysis

Local neurotransmitter fingerprinting scores the z-scored PET atlas values directly within the lesion mask. For each neurotransmitter target, it computes the mean (or sum) of the atlas values at lesion voxels.

This answers: **what neurotransmitter landscape did the lesion wipe out?**

A high score for a given target indicates that the lesioned region is rich in that neurotransmitter system, suggesting potential neurochemical consequences of the lesion.

Run the analysis.

In [44]:
!lacuna run lntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_lntf/ \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_data

In [46]:
!lacuna run lntf --help

usage: lacuna run lntf [-h] [--participant-label LABEL [LABEL ...]]
                       [--session-id SESSION [SESSION ...]] [--pattern GLOB]
                       [--mask-space SPACE] [--overwrite]
                       [--on-empty {warn,skip,error}] [--keep-intermediate]
                       [-v]
                       (--atlas-cache-dir ATLAS_CACHE_DIR | --ace-cache-dir ACE_CACHE_DIR)
                       [--aggregation {mean,sum}]
                       bids_dir output_dir

Local neurotransmitter fingerprinting: score NT atlas values directly
within the lesion mask.

Examples:
  lacuna run lntf /bids /output --atlas-cache-dir /path/to/ntatlas
  lacuna run lntf /bids /output --ace-cache-dir /path/to/ace

positional arguments:
  bids_dir              Root folder of BIDS dataset (sub-XXXXX folders at top
                        level), OR path to a single NIfTI mask file for quick
                        analysis
  output_dir            Output directory for derivatives

optio

List the outputs.

In [47]:
!ls /tmp/outputs_lntf/sub-01/ses-01/anat/

sub-01_ses-01_label-acuteinfarct_atlas-neurotransmitter_desc-lntfscores_parcelstats.tsv
sub-01_ses-01_label-acuteinfarct_method-lntf_atlas-neurotransmitter_desc-static_labelstats.json
sub-01_ses-01_label-acuteinfarct_method-lntf_atlas-neurotransmitter_desc-static_labelstats.tsv


Visualize the per-target neurotransmitter scores.

In [51]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# If scores are in TSV format instead
import pandas as pd
tsv_files = sorted(results_dir.glob("*labelstats.tsv"))

if tsv_files:
    df = pd.read_csv(tsv_files[0], sep="\t")
df

,target,value
0,5HT1a,-0.245860
1,5HT1b,1.138475
2,5HT2a,0.764246
3,5HT4,-0.182223
4,5HT6,0.405534
5,5HTT,0.487374
6,A4B2,0.262904
7,CB1,-0.387774
8,D1,0.449205
9,D23,-0.087813


## Run on multiple subjects

Lacuna supports processing multiple subjects within a single run. If the `--participant-label` flag is omitted, the pipeline automatically processes all subjects detected in the BIDS dataset.

LNTF is very fast since it only requires voxel lookups in the atlas — no connectome loading is needed.

In [ ]:
!lacuna run lntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_lntf_all/ \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_data

Collect results into a group-level table.

In [ ]:
!lacuna collect \
    /tmp/outputs_lntf_all/ \
    --pattern "*lntf*parcelstats*" \
    --output-dir /tmp/group_lntm/

In [ ]:
!ls /tmp/group_lntm/